**1. Setup and configuration**

In [0]:
import sys
from pathlib import Path
from pyspark.sql import functions as F

REPO_PATH = Path("/Workspace/Users/tiiina.aminian@gmail.com/Ontario_Electricity_Lakehouse")
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

from src.ingestion.config import IESO_DEMAND_2025
from src.ingestion.bronze_loader import read_stream, prepare_bronze_stream
from src.ingestion.bronze_writer import write_bronze_stream
from src.transformation.silver_demand import transform_silver_demand
from src.transformation.silver_validation import validate_silver_demand
from src.transformation.silver_writer import write_silver

BRONZE_TABLE = "ontario_energy.bronze.ieso_demand_2025"
SILVER_TABLE = "ontario_energy.silver.ieso_demand_2025"
CHECKPOINT_PATH = "/Volumes/ontario_energy/landing/checkpoints"
SOURCE_CSV = "/Volumes/ontario_energy/landing/source_file/PUB_Demand_2025.csv"
RUN_BRONZE_INGESTION = False  # Set True only when intentionally running Auto Loader.

print("Repository:", REPO_PATH)
print("Bronze ingestion enabled:", RUN_BRONZE_INGESTION)

**2. Bronze ingestion**

In [0]:
if RUN_BRONZE_INGESTION:
    source_stream = read_stream(spark, IESO_DEMAND_2025)
    bronze_stream = prepare_bronze_stream(source_stream)

    query = write_bronze_stream(
        df=bronze_stream,
        target_table=BRONZE_TABLE,
        checkpoint_path=CHECKPOINT_PATH,
    )
    query.awaitTermination()
    print("Bronze ingestion completed.")
else:
    print("Bronze ingestion skipped; using the existing Bronze Delta table.")

**3. Inspect bronze records**

Inspect the raw schema, record, record count, and date/hour coverage before transformation

In [0]:
bronze_df = spark.table(BRONZE_TABLE)
bronze_count = bronze_df.count()

print(f"Bronze records: {bronze_count:,}")
bronze_df.printSchema()
display(bronze_df.limit(10))

bronze_df.select(
    F.min("date_raw").alias("first_date"),
    F.max("date_raw").alias("last_date"),
    F.countDistinct("date_raw").alias("distinct_dates"),
    F.min(F.col("hour_raw").cast("int")).alias("min_hour"),
    F.max(F.col("hour_raw").cast("int")).alias("max_hour"),
).show()

**4. Bronze data-quality profile**

check missing/blank source values, duplicates and unexpected hour counts.

In [0]:
raw_columns = ["date_raw", "hour_raw", "market_demand_raw", "ontario_demand_raw"]

missing_values = bronze_df.select(*[
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in raw_columns
])
print("Null/blank source values:")
missing_values.show()

print("Unparseable numeric values:")
bronze_df.select(
    F.sum(F.when(F.expr("try_cast(hour_raw AS INT)").isNull(), 1).otherwise(0)).alias("invalid_hour"),
    F.sum(F.when(F.expr("try_cast(market_demand_raw AS DECIMAL(12,2))").isNull(), 1).otherwise(0)).alias("invalid_market_demand"),
    F.sum(F.when(F.expr("try_cast(ontario_demand_raw AS DECIMAL(12,2))").isNull(), 1).otherwise(0)).alias("invalid_ontario_demand"),
).show()

duplicate_hours = (
    bronze_df.groupBy("date_raw", "hour_raw")
    .count().filter(F.col("count") > 1)
)
print("Duplicate date-hour keys:", duplicate_hours.count())

daily_counts = bronze_df.groupBy("date_raw").agg(F.count("*").alias("hour_count"))
incomplete_days = daily_counts.filter(F.col("hour_count") != 24)
print("Days with an unexpected number of rows:")
display(incomplete_days.orderBy("date_raw"))

**Source investigation for May 1, 2025**

In [0]:
from pyspark.sql import functions as F

source_df = (
    spark.read
    .option("header", "true")
    .option("comment", "\\")
    .csv("/Volumes/ontario_energy/landing/source_file/PUB_Demand_2025.csv")
)

source_df.filter(
    (F.col("Date") == "2025-05-01") &
    (F.col("Hour") == "1")
).show(truncate=False)


**5. Transform bronze -> silver**

converting source strings into typed data, hour-ending integer and decimal MW measurements. Retaining source-file lineage and ingestion timestamps.

In [0]:

silver_df = transform_silver_demand(bronze_df)
silver_df.printSchema()

**Validate data quality**

validating silver before writing.

In [0]:
silver_count = silver_df.count()
assert silver_count == bronze_count, (
    f"Bronze/Silver row count mismatch: {bronze_count} vs {silver_count}"
)

validate_silver_demand(silver_df)
print(f"PASS: {silver_count:,} Silver rows validated")

**7. Writing data into silver delta table**

Full-refresh strategy

**8. write to the silver table**

In [0]:
write_silver(silver_df, target_table=SILVER_TABLE)
print("silver write completed")

**9. validate the persisted Delta table**

In [0]:
persisted_silver_df = spark.table(SILVER_TABLE)
persisted_count = persisted_silver_df.count()
assert persisted_count == silver_count, (
    f"Persisted count mismatch: expected {silver_count}, found {persisted_count}"
)
print(f"PASS: {persisted_count:,} records persisted to {SILVER_TABLE}")
persisted_silver_df.printSchema()
display(persisted_silver_df.orderBy("demand_date", "hour_ending").limit(10))

**10. Gold transformation**

In [0]:
from src.transformation.gold_demand import transform_daily_demand
from src.transformation.gold_validation import validate_daily_demand
from src.transformation.gold_writer import write_gold_daily_demand

#Extract
silver_df = spark.table("ontario_energy.silver.ieso_demand_2025")
GOLD_TABLE = "ontario_energy.gold.daily_demand_summary"

#Validate
gold_daily_demand_df = transform_daily_demand(silver_df)
validate_daily_demand(gold_daily_demand_df)

#Load
write_gold_daily_demand(silver_df, GOLD_TABLE)
